In [ ]:
# 1. Mount Google Drive
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/My Drive/ColabNotebooks/I5/ADV')
SOURCE = DATA_DIR / 'dataasilida_cleaned.csv'
assert SOURCE.is_file(), f"Input file not found: {SOURCE}"
print('Source:', SOURCE)
print('Size (bytes):', SOURCE.stat().st_size)

Mounted at /content/drive
Source: /content/drive/My Drive/ColabNotebooks/I5/ADV/dataasilida_cleaned.csv
Size (bytes): 581378


In [ ]:
# 2. Read original CSV
import csv
import pandas as pd

with SOURCE.open('r', encoding='utf-8-sig', newline='') as f:
    raw_rows = list(csv.reader(f))

print('Original records:', len(raw_rows) - 1)

Original records: 521


In [ ]:
# 3. Repair malformed records
import re
import unicodedata

EXPECTED_COLUMNS = ['input', 'output']
header = raw_rows[0]
assert header in (["input,output"], ["input", "output"]), f"Unexpected header: {header}"

fixed_records = []
repaired_row_numbers = []
rejected_records = []

for line_number, fields in enumerate(raw_rows[1:], start=2):
    if len(fields) == 2:
        question, answer = fields
    elif len(fields) == 1 and ',' in fields[0]:
        # The malformed examples have no enclosing field quotes.
        # Only split on the first comma, preserving commas within the answer.
        question, answer = fields[0].split(',', 1)
        repaired_row_numbers.append(line_number)
    elif len(fields) == 1 and '?' in fields[0]:
        question, answer = fields[0].split('?', 1)
        question = question.strip() + '?'
        answer = answer.strip()
        if answer.startswith(r'\n'):
            answer = answer[2:].strip()
        repaired_row_numbers.append(line_number)
    else:
        rejected_records.append((line_number, fields))
        continue
    fixed_records.append((question, answer))

print('Repaired rows:', repaired_row_numbers)
print('Unrecoverable rows:', len(rejected_records))
if rejected_records:
    for n, fields in rejected_records[:5]:
        print('Inspect record', n, repr(fields)[:200])
assert not rejected_records, 'Review unrecoverable records before proceeding.'

Repaired rows: [98, 99, 101, 393, 395, 419, 424]
Unrecoverable rows: 0


In [ ]:
# 4. Clean text, remove empty and duplicate pairs
import pandas as pd

raw_df = pd.DataFrame(fixed_records, columns=EXPECTED_COLUMNS)

def clean_text(value):
    if pd.isna(value):
        return ''
    # Normalize Unicode encoding consistently; do not translate Khmer or remove punctuation.
    value = unicodedata.normalize('NFC', str(value))
    return re.sub(r'\s+', ' ', value).strip()

df = raw_df.copy()
for column in EXPECTED_COLUMNS:
    df[column] = df[column].map(clean_text)

blank_mask = df['input'].eq('') | df['output'].eq('')
blank_count = int(blank_mask.sum())
df = df.loc[~blank_mask].copy()
duplicate_count = int(df.duplicated(subset=EXPECTED_COLUMNS).sum())
df = df.drop_duplicates(subset=EXPECTED_COLUMNS, keep='first').reset_index(drop=True)

print('Pairs parsed:', len(raw_df))
print('Blank Q&A removed:', blank_count)
print('Exact duplicate Q&A removed:', duplicate_count)
print('Final clean pairs:', len(df))
display(df.head(5))

Pairs parsed: 521
Blank Q&A removed: 0
Exact duplicate Q&A removed: 2
Final clean pairs: 519


,input,output
0,តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ជាអ្វី,ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា គឺជាប័ណ្ណ...
1,តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ផ្តល់នូ...,លោកអ្នកទទួលបានអត្ថប្រយោជន៍ពីការប្រើប្រាស់ ប័ណ្...
2,តើធ្វើដូចម្តេចទើបខ្ញុំអាច ស្នើសុំប័ណ្ណឥណពន្ធចេ...,លោកអ្នកអាចស្នើសុំ ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដ...
3,តើខ្ញុំអាចប្រើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ដក...,លោកអ្នកអាចដកប្រាក់បានរហូតដល់ ១០០០០ដុល្លារអាមេរ...
4,តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា មានសុពល...,ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា មានសុពលភា...


In [ ]:
# 5. Validate and export
# Validate and save the only output file.
assert list(df.columns) == ['input', 'output']
assert len(df) > 0
assert df.notna().all().all()
assert df.ne('').all().all()
assert not df.duplicated().any()

output_path = DATA_DIR / 'data_kh.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')

# Verify the saved CSV can be loaded correctly.
check = pd.read_csv(output_path, encoding='utf-8-sig', keep_default_na=False)
pd.testing.assert_frame_equal(df, check)
print('PASS: cleaned dataset saved successfully.')
print('Shape:', check.shape)
print('Saved to:', output_path)

PASS: cleaned dataset saved successfully.
Shape: (519, 2)
Saved to: /content/drive/My Drive/ColabNotebooks/I5/ADV/data_kh.csv
